In [2]:
import json
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split
import pandas as pd
from collections import defaultdict
import numpy as np

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path

# Define the path to the current data chunk directory
chunk_dir = Path("/content/drive/MyDrive/DL_project_finetuned/sample_data")
out_dir = Path("/content/drive/MyDrive/DL_project_finetuned/sample_data/oragnised")

In [7]:
import shutil
from pathlib import Path

extraction_dir = Path("/content/drive/MyDrive/DL_project_finetuned/sample_data/extracted_chunk")

if extraction_dir.exists():
    print(f"Deleting extraction directory: {extraction_dir}")
    shutil.rmtree(extraction_dir)
    print("Extraction directory deleted.")
else:
    print("Extraction directory does not exist.")

Deleting extraction directory: /content/drive/MyDrive/DL_project_finetuned/sample_data/extracted_chunk
Extraction directory deleted.


In [9]:
def prepare_dfdc_data_splits(
    source_data_dir,
    output_dir,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    random_state=42
):
    """
    Prepare DFDC data in the required format, appending to existing data if output_dir exists:
    data/
      train/real_videos/*.mp4
      train/fake_videos/*.mp4
      val/real_videos/*.mp4
      val/fake_videos/*.mp4
      test/real_videos/*.mp4
      test/fake_videos/*.mp4

    Args:
        source_data_dir: Path to the original DFDC data directory (for a single chunk)
        output_dir: Path where organized data will be saved (and appended to)
        train_ratio: Proportion of data for training (default: 0.8) - applies to the *new* chunk
        val_ratio: Proportion of data for validation (default: 0.1) - applies to the *new* chunk
        test_ratio: Proportion of data for testing (default: 0.1) - applies to the *new* chunk
        random_state: Random seed for reproducible splits (used for splitting the *new* chunk)
    """

    # Verify ratios sum to 1
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6, "Ratios must sum to 1.0"

    # Paths
    source_videos_dir = Path(source_data_dir)
    metadata_path = source_videos_dir / "metadata.json" # Modified line to look inside the chunk folder
    output_path = Path(output_dir)
    summary_path = output_path / 'data_split_summary.json'

    # Load existing summary and data info if output_dir exists
    existing_summary = None
    existing_video_paths = defaultdict(lambda: defaultdict(list))

    if output_path.exists() and summary_path.exists():
        print(f"Existing organized data found at {output_path}. Appending new data.")
        try:
            with open(summary_path, 'r') as f:
                existing_summary = json.load(f)
            # Reconstruct existing video paths from directory structure
            for split in ['train', 'val', 'test']:
                for label_dir in ['real_videos', 'fake_videos']:
                    split_dir = output_path / split / label_dir
                    if split_dir.exists():
                        existing_video_paths[split][label_dir].extend(list(split_dir.glob('*.mp4')))
        except Exception as e:
            print(f"Error loading existing data or summary: {e}. Starting fresh.")
            existing_summary = None
            existing_video_paths = defaultdict(lambda: defaultdict(list))
            # Ensure output directories exist even if starting fresh due to error
            for split_name in ['train', 'val', 'test']:
                 for category in ['real_videos', 'fake_videos']:
                    (output_path / split_name / category).mkdir(parents=True, exist_ok=True)
    else:
        print(f"No existing organized data found at {output_path}. Creating new structure.")
        # Create directory structure for the first time
        for split_name in ['train', 'val', 'test']:
             for category in ['real_videos', 'fake_videos']:
                (output_path / split_name / category).mkdir(parents=True, exist_ok=True)


    # Load new chunk's metadata
    print(f"Loading metadata for the new chunk from: {metadata_path}")
    if not metadata_path.exists():
        print(f"Error: Metadata file not found for the new chunk at {metadata_path}")
        return existing_summary if existing_summary else {} # Return existing summary or empty if error

    with open(metadata_path, 'r') as f:
        new_metadata = json.load(f)

    # Organize new videos by label
    new_real_videos = []
    new_fake_videos = []

    # The source_videos_dir is still needed to find the actual video files
    # source_videos_dir is now the source_data_dir itself

    for video_file, info in new_metadata.items():
        video_full_path = source_videos_dir / video_file
        if video_full_path.exists(): # Only process if the video file actually exists
            if info['label'] == 'REAL':
                new_real_videos.append(video_file)
            elif info['label'] == 'FAKE':
                new_fake_videos.append(video_file)
        else:
            print(f"Warning: Video file not found in source: {video_full_path}")


    print(f"Found {len(new_real_videos)} REAL videos in the new chunk")
    print(f"Found {len(new_fake_videos)} FAKE videos in the new chunk")

    # Function to split data stratified by label (for the new chunk)
    def stratified_split_new_chunk(videos, train_r, val_r, test_r, random_state):
        if not videos:
            return [], [], []
        # First split: train vs (val + test)
        train_videos, temp_videos = train_test_split(
            videos,
            test_size=(val_r + test_r),
            random_state=random_state
        )

        if not temp_videos: # Handle cases where only train data is left
             return train_videos, [], []

        # Second split: val vs test from the temp set
        val_videos, test_videos = train_test_split(
            temp_videos,
            test_size=test_r / (val_r + test_r),  # Adjust ratio for the remaining data
            random_state=random_state
        )

        return train_videos, val_videos, test_videos

    # Split new real and fake videos separately
    new_real_train, new_real_val, new_real_test = stratified_split_new_chunk(
        new_real_videos, train_ratio, val_ratio, test_ratio, random_state
    )
    new_fake_train, new_fake_val, new_fake_test = stratified_split_new_chunk(
        new_fake_videos, train_ratio, val_ratio, test_ratio, random_state
    )

    # Print new chunk split statistics
    print(f"\\nNew Chunk Split Statistics:")
    print(f"REAL videos - Train: {len(new_real_train)}, Val: {len(new_real_val)}, Test: {len(new_real_test)}")
    print(f"FAKE videos - Train: {len(new_fake_train)}, Val: {len(new_fake_val)}, Test: {len(new_fake_test)}")
    print(f"Total - Train: {len(new_real_train) + len(new_fake_train)}, Val: {len(new_real_val) + len(new_fake_val)}, Test: {len(new_real_test) + len(new_fake_test)}")

    # Append new videos to existing lists (if any)
    current_splits = {
        'train': {'real_videos': [p.name for p in existing_video_paths['train']['real_videos']] + new_real_train,
                  'fake_videos': [p.name for p in existing_video_paths['train']['fake_videos']] + new_fake_train},
        'val': {'real_videos': [p.name for p in existing_video_paths['val']['real_videos']] + new_real_val,
                'fake_videos': [p.name for p in existing_video_paths['val']['fake_videos']] + new_fake_val},
        'test': {'real_videos': [p.name for p in existing_video_paths['test']['real_videos']] + new_real_test,
                 'fake_videos': [p.name for p in existing_video_paths['test']['fake_videos']] + new_fake_test}
    }


    # Create directories and copy new files
    for split_name, categories in current_splits.items():
        for category, video_list in categories.items():
            target_dir = output_path / split_name / category
            # target_dir.mkdir(parents=True, exist_ok=True) # Directories are created initially or if error

            print(f"\\nCopying new videos for {split_name}/{category} to {target_dir}")

            # Identify which videos are new by checking if they already exist in the target directory
            existing_files_in_target = set([p.name for p in target_dir.glob('*.mp4')])
            newly_added_count = 0

            for video_file in video_list:
                # The source file path needs to be constructed using the original source_data_dir
                source_file = Path(source_data_dir) / video_file
                target_file = target_dir / video_file

                if video_file not in existing_files_in_target:
                     if source_file.exists():
                        shutil.copy2(source_file, target_file)
                        newly_added_count += 1
                     else:
                        print(f"Warning: Source file not found for copying: {source_file}")
                # else:
                #     print(f"Info: File already exists in target, skipping: {target_file}") # Optional: uncomment for detailed logging

            print(f"Copied {newly_added_count} new videos to {target_dir}")


    # Create updated summary metadata
    updated_summary = {
        'splits': {
            'train': {
                'real_count': len(current_splits['train']['real_videos']),
                'fake_count': len(current_splits['train']['fake_videos']),
                'total': len(current_splits['train']['real_videos']) + len(current_splits['train']['fake_videos'])
            },
            'val': {
                'real_count': len(current_splits['val']['real_videos']),
                'fake_count': len(current_splits['val']['fake_videos']),
                'total': len(current_splits['val']['real_videos']) + len(current_splits['val']['fake_videos'])
            },
            'test': {
                'real_count': len(current_splits['test']['real_videos']),
                'fake_count': len(current_splits['test']['fake_videos']),
                'total': len(current_splits['test']['real_videos']) + len(current_splits['test']['fake_videos'])
            }
        },
        'ratios': {
            'train': train_ratio,
            'val': val_ratio,
            'test': test_ratio
        },
        'random_state': random_state,
        'processed_chunks': (existing_summary.get('processed_chunks', 0) if existing_summary else 0) + 1 # Track processed chunks
    }

    # Save updated summary
    with open(summary_path, 'w') as f:
        json.dump(updated_summary, f, indent=2)

    print(f"\\nData organization complete!")
    print(f"Organized data saved to: {output_path}")
    print(f"Updated summary saved to: {summary_path}")

    return updated_summary

In [10]:
from pathlib import Path

# Define the path to the zip file for the chunk you want to process
# Construct the zip file path dynamically based on the chunk_dir
# Assuming the zip file name format is like dfdc_train_part_XX.zip
chunk_number = chunk_dir.name.split(" ")[-1] # Extract chunk number from directory name
zip_file_name = f"dfdc_train_part_{chunk_number}.zip"
chunk_zip_path = chunk_dir / zip_file_name

print(f"Chunk zip path defined: {chunk_zip_path}")

Chunk zip path defined: /content/drive/MyDrive/DL_project_finetuned/sample_data/dfdc_train_part_sample_data.zip


In [11]:
import os
from pathlib import Path

# Define the extraction directory (e.g., based on the zip file name)
extraction_dir = Path("/content/extracted_chunk") # You can change this path if needed
# Use exist_ok=True to avoid errors if the directory already exists
extraction_dir.mkdir(parents=True, exist_ok=True)

print(f"Extraction directory created or already exists: {extraction_dir}")

Extraction directory created or already exists: /content/extracted_chunk


In [12]:
# Assuming the zip file name is the last part of the chunk_zip_path
# You might need to adjust the zip file name if it's different
zip_file_name = chunk_zip_path.name
!unzip "{chunk_zip_path}" -d "{extraction_dir}"

unzip:  cannot find or open /content/drive/MyDrive/DL_project_finetuned/sample_data/dfdc_train_part_sample_data.zip, /content/drive/MyDrive/DL_project_finetuned/sample_data/dfdc_train_part_sample_data.zip.zip or /content/drive/MyDrive/DL_project_finetuned/sample_data/dfdc_train_part_sample_data.zip.ZIP.


In [13]:
import os
from pathlib import Path

extracted_dir_path = Path(f"/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos")

if extracted_dir_path.exists():
    print(f"Listing contents of: {extracted_dir_path}")
    # List all files and directories in the extracted folder
    for item in extracted_dir_path.iterdir():
        print(item)

    # You can also check for the metadata file specifically
    metadata_path_in_extracted = extracted_dir_path / "metadata.json"
    print(f"\nChecking for metadata.json at: {metadata_path_in_extracted}")
    if metadata_path_in_extracted.exists():
        print("metadata.json found directly in the extracted directory.")
    else:
        print("metadata.json NOT found directly in the extracted directory.")

else:
    print(f"Error: Extracted directory not found at {extracted_dir_path}")

Listing contents of: /content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/aagfhgtpmv.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/aapnvogymq.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/acifjvzvpm.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/abofeumbvv.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/acxnxvbsxk.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/abqwwspghj.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/abarnvbtwb.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/acqfdwsrhi.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/aczrgyricp.mp4
/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/adhsbajy

In [15]:
extracted_chunk_subdir = Path("/content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos") # Correct path to the extracted subdirectory
print(f"Processing extracted data from: {extracted_chunk_subdir}")
summary_chunk_extracted = prepare_dfdc_data_splits(source_data_dir=str(extracted_chunk_subdir), output_dir=str(out_dir))

print("\nSummary after processing extracted chunk:")
print(json.dumps(summary_chunk_extracted, indent=2))

Processing extracted data from: /content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos
No existing organized data found at /content/drive/MyDrive/DL_project_finetuned/sample_data/oragnised. Creating new structure.
Loading metadata for the new chunk from: /content/drive/MyDrive/DL_project_finetuned/sample_data/train_sample_videos/metadata.json
Found 77 REAL videos in the new chunk
Found 323 FAKE videos in the new chunk
\nNew Chunk Split Statistics:
REAL videos - Train: 61, Val: 8, Test: 8
FAKE videos - Train: 258, Val: 32, Test: 33
Total - Train: 319, Val: 40, Test: 41
\nCopying new videos for train/real_videos to /content/drive/MyDrive/DL_project_finetuned/sample_data/oragnised/train/real_videos
Copied 61 new videos to /content/drive/MyDrive/DL_project_finetuned/sample_data/oragnised/train/real_videos
\nCopying new videos for train/fake_videos to /content/drive/MyDrive/DL_project_finetuned/sample_data/oragnised/train/fake_videos
Copied 258 new videos to /content/dr

In [16]:
# Additional utility functions for working with the organized data

def load_organized_data_info(organized_data_dir):
    """
    Load information about the organized data splits
    """
    organized_path = Path(organized_data_dir)
    summary_path = organized_path / 'data_split_summary.json'

    if summary_path.exists():
        with open(summary_path, 'r') as f:
            summary = json.load(f)
        return summary
    else:
        print(f"Summary file not found at {summary_path}")
        return None

def get_video_paths_by_split(organized_data_dir):
    """
    Get all video paths organized by split and label

    Returns:
        dict: Nested dictionary with structure:
              {split: {label: [list_of_video_paths]}}
    """
    organized_path = Path(organized_data_dir)
    video_paths = {}

    for split in ['train', 'val', 'test']:
        video_paths[split] = {}
        for label in ['real_videos', 'fake_videos']:
            split_dir = organized_path / split / label
            if split_dir.exists():
                video_paths[split][label] = list(split_dir.glob('*.mp4'))
            else:
                video_paths[split][label] = []

    return video_paths

def create_dataframe_from_organized_data(organized_data_dir):
    """
    Create a pandas DataFrame with video paths and labels for easy manipulation

    Returns:
        pd.DataFrame: DataFrame with columns ['video_path', 'label', 'split', 'filename']
    """
    video_paths = get_video_paths_by_split(organized_data_dir)

    data = []
    for split, labels in video_paths.items():
        for label_dir, paths in labels.items():
            # Convert label_dir to simple label
            label = 'REAL' if label_dir == 'real_videos' else 'FAKE'

            for path in paths:
                data.append({
                    'video_path': str(path),
                    'label': label,
                    'split': split,
                    'filename': path.name
                })

    return pd.DataFrame(data)

def verify_data_organization(organized_data_dir):
    """
    Verify the data organization and print statistics
    """
    print("Verifying data organization...")

    # Load summary
    summary = load_organized_data_info(organized_data_dir)
    if summary:
        print("\\nSummary from metadata:")
        for split, counts in summary['splits'].items():
            print(f"  {split.upper()}: {counts['real_count']} real, {counts['fake_count']} fake, {counts['total']} total")

    # Count actual files
    video_paths = get_video_paths_by_split(organized_data_dir)
    print("\\nActual file counts:")

    total_real = 0
    total_fake = 0

    for split, labels in video_paths.items():
        real_count = len(labels['real_videos'])
        fake_count = len(labels['fake_videos'])
        total_count = real_count + fake_count

        total_real += real_count
        total_fake += fake_count

        print(f"  {split.upper()}: {real_count} real, {fake_count} fake, {total_count} total")

    print(f"\\nOverall totals: {total_real} real, {total_fake} fake, {total_real + total_fake} total")

    # Create and display DataFrame
    df = create_dataframe_from_organized_data(organized_data_dir)
    print(f"\\nDataFrame shape: {df.shape}")
    print("\\nDataFrame sample:")
    print(df.head())

    print("\\nLabel distribution by split:")
    print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

    return df

# Run verification if the organized data exists
if Path(out_dir).exists():
    df = verify_data_organization(out_dir)
else:
    print(f"Organized data directory not found: {out_dir}")
    print("Run the first cell to organize the data first.")


Verifying data organization...
\nSummary from metadata:
  TRAIN: 61 real, 258 fake, 319 total
  VAL: 8 real, 32 fake, 40 total
  TEST: 8 real, 33 fake, 41 total
\nActual file counts:
  TRAIN: 61 real, 258 fake, 319 total
  VAL: 8 real, 32 fake, 40 total
  TEST: 8 real, 33 fake, 41 total
\nOverall totals: 77 real, 323 fake, 400 total
\nDataFrame shape: (400, 4)
\nDataFrame sample:
                                          video_path label  split  \
0  /content/drive/MyDrive/DL_project_finetuned/sa...  REAL  train   
1  /content/drive/MyDrive/DL_project_finetuned/sa...  REAL  train   
2  /content/drive/MyDrive/DL_project_finetuned/sa...  REAL  train   
3  /content/drive/MyDrive/DL_project_finetuned/sa...  REAL  train   
4  /content/drive/MyDrive/DL_project_finetuned/sa...  REAL  train   

         filename  
0  atvmxvwyns.mp4  
1  ajqslcypsw.mp4  
2  cfxkpiweqt.mp4  
3  bgwmmujlmc.mp4  
4  bxzakyopjf.mp4  
\nLabel distribution by split:
label  FAKE  REAL
split            
test     33    